In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/alexandr2017/training-set/training_set.jsonl
/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl


# Експеримент 2 — Fine-tuning з Validation Set
## Покращення urgency accuracy, особливо для 'medium'

In [2]:
# ────────────────────────────────────────────────────────────
# 1: Встановлення + середовище
# (якщо вже встановлено в тій самій сесії — пропускай)
# ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q",
    "unsloth[kaggle-new]", "unsloth_zoo",
    "bitsandbytes", "trl", "peft",
    "torchao>=0.16.0", "cut_cross_entropy",
    "hf_transfer", "msgspec",
    "--upgrade"
], check=False)
 
import os, json, re, hashlib, torch
from collections import defaultdict, Counter
from kaggle_secrets import UserSecretsClient
from unsloth import FastLanguageModel, is_bfloat16_supported, UnslothTrainer, UnslothTrainingArguments
from datasets import Dataset
 
secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
 
BASE_DIR    = '/kaggle/working/lesson17_finetune_v2'
ADAPTER_V1  = '/kaggle/working/lesson17_finetune/lora_adapter'  # адаптер з exp1
os.makedirs(BASE_DIR, exist_ok=True)
 
EVAL_PATH  = '/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl'
TRAIN_PATH = '/kaggle/input/datasets/alexandr2017/training-set/training_set.jsonl'
 
SYSTEM_PROMPT = """You are a support email classifier. Extract information from the email and return ONLY a valid JSON object with exactly these fields:
- sender: full name of the sender (null if anonymous)
- product: one of [Payments, Inventory, Analytics, Storefront, Shipping, Subscriptions]
- category: one of [billing, technical, account, feature_request, other]
- urgency: one of [low, medium, high, critical]
- summary: one sentence summary of the issue
 
Return ONLY the JSON object, no explanation, no markdown, no extra text."""
 
def build_prompt(email_text):
    return f"{SYSTEM_PROMPT}\n\nEmail:\n{email_text}\n\nJSON:"
 
def parse_json_response(raw):
    raw = raw.strip()
    try: return json.loads(raw)
    except: pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return None
 
def score_prediction(pred, expected):
    if pred is None:
        return {k: 0 for k in ['json_valid','sender_correct','product_correct',
                                'category_correct','urgency_correct',
                                'summary_present','exact_match']}
    sender_correct   = (
        (pred.get('sender') is None and expected.get('sender') is None) or
        (str(pred.get('sender','')).strip().lower() ==
         str(expected.get('sender','')).strip().lower())
    )
    product_correct  = pred.get('product')  == expected.get('product')
    category_correct = pred.get('category') == expected.get('category')
    urgency_correct  = pred.get('urgency')  == expected.get('urgency')
    summary_present  = bool(pred.get('summary','').strip())
    exact_match      = all([sender_correct, product_correct,
                            category_correct, urgency_correct, summary_present])
    return {
        'json_valid'       : 1,
        'sender_correct'   : int(sender_correct),
        'product_correct'  : int(product_correct),
        'category_correct' : int(category_correct),
        'urgency_correct'  : int(urgency_correct),
        'summary_present'  : int(summary_present),
        'exact_match'      : int(exact_match),
    }
 
print("✅ Середовище готове")
 


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Середовище готове


In [3]:
# ────────────────────────────────────────────────────────────
# 2: Підготовка train/validation split
# Той самий training_set.jsonl що і в експерименті 1
# 80/20 стратифікований split по urgency
# ────────────────────────────────────────────────────────────
import random
random.seed(42)
 
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    all_train = [json.loads(line) for line in f]
 
print(f"Загалом прикладів: {len(all_train)}")
 
# Стратифікований split — берем 20% з кожного urgency рівня
by_urgency = defaultdict(list)
for item in all_train:
    by_urgency[item['expected']['urgency']].append(item)
 
train_split = []
val_split   = []
 
for urgency, items in by_urgency.items():
    random.shuffle(items)
    val_size = max(1, int(len(items) * 0.2))
    val_split.extend(items[:val_size])
    train_split.extend(items[val_size:])
 
print(f"\n📊 SPLIT РЕЗУЛЬТАТ:")
print(f"   Training   : {len(train_split)} прикладів")
print(f"   Validation : {len(val_split)} прикладів")
 
train_urg = Counter(item['expected']['urgency'] for item in train_split)
val_urg   = Counter(item['expected']['urgency'] for item in val_split)
print(f"\n   Training urgency  : {dict(train_urg)}")
print(f"   Validation urgency: {dict(val_urg)}")
 
# Hash-check — val не перетинається з eval set
eval_hashes_path = '/kaggle/working/lesson17_finetune/eval_hashes.json'
if os.path.exists(eval_hashes_path):
    with open(eval_hashes_path) as f:
        eval_hashes = set(json.load(f).keys())
    overlaps = sum(1 for item in val_split
                   if hashlib.md5(item['email'].strip().encode()).hexdigest() in eval_hashes)
    print(f"\n   Overlap val↔eval  : {overlaps} (має бути 0)")
else:
    print(f"\n   ⚠️ eval_hashes.json не знайдено — пропускаємо hash-check")
 


Загалом прикладів: 300

📊 SPLIT РЕЗУЛЬТАТ:
   Training   : 241 прикладів
   Validation : 59 прикладів

   Training urgency  : {'critical': 50, 'high': 82, 'low': 60, 'medium': 49}
   Validation urgency: {'critical': 12, 'high': 20, 'low': 15, 'medium': 12}

   ⚠️ eval_hashes.json не знайдено — пропускаємо hash-check


# LESSON 17: Fine-tuning з Validation Set

In [4]:
# ────────────────────────────────────────────────────────────
# 1: Встановлення + середовище
# (якщо вже встановлено в тій самій сесії — пропускай)
# ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q",
    "unsloth[kaggle-new]", "unsloth_zoo",
    "bitsandbytes", "trl", "peft",
    "torchao>=0.16.0", "cut_cross_entropy",
    "hf_transfer", "msgspec",
    "--upgrade"
], check=False)
 
import os, json, re, hashlib, torch
from collections import defaultdict, Counter
from kaggle_secrets import UserSecretsClient
from unsloth import FastLanguageModel, is_bfloat16_supported, UnslothTrainer, UnslothTrainingArguments
from datasets import Dataset
 
secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
 
BASE_DIR    = '/kaggle/working/lesson17_finetune_v2'
ADAPTER_V1  = '/kaggle/working/lesson17_finetune/lora_adapter'  # адаптер з exp1
os.makedirs(BASE_DIR, exist_ok=True)
 
EVAL_PATH  = '/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl'
TRAIN_PATH = '/kaggle/input/datasets/alexandr2017/training-set/training_set.jsonl'
 
SYSTEM_PROMPT = """You are a support email classifier. Extract information from the email and return ONLY a valid JSON object with exactly these fields:
- sender: full name of the sender (null if anonymous)
- product: one of [Payments, Inventory, Analytics, Storefront, Shipping, Subscriptions]
- category: one of [billing, technical, account, feature_request, other]
- urgency: one of [low, medium, high, critical]
- summary: one sentence summary of the issue
 
Return ONLY the JSON object, no explanation, no markdown, no extra text."""
 
def build_prompt(email_text):
    return f"{SYSTEM_PROMPT}\n\nEmail:\n{email_text}\n\nJSON:"
 
def parse_json_response(raw):
    raw = raw.strip()
    try: return json.loads(raw)
    except: pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return None
 
def score_prediction(pred, expected):
    if pred is None:
        return {k: 0 for k in ['json_valid','sender_correct','product_correct',
                                'category_correct','urgency_correct',
                                'summary_present','exact_match']}
    sender_correct   = (
        (pred.get('sender') is None and expected.get('sender') is None) or
        (str(pred.get('sender','')).strip().lower() ==
         str(expected.get('sender','')).strip().lower())
    )
    product_correct  = pred.get('product')  == expected.get('product')
    category_correct = pred.get('category') == expected.get('category')
    urgency_correct  = pred.get('urgency')  == expected.get('urgency')
    summary_present  = bool(pred.get('summary','').strip())
    exact_match      = all([sender_correct, product_correct,
                            category_correct, urgency_correct, summary_present])
    return {
        'json_valid'       : 1,
        'sender_correct'   : int(sender_correct),
        'product_correct'  : int(product_correct),
        'category_correct' : int(category_correct),
        'urgency_correct'  : int(urgency_correct),
        'summary_present'  : int(summary_present),
        'exact_match'      : int(exact_match),
    }
 
print("✅ Середовище готове")
 


✅ Середовище готове


In [5]:
# ────────────────────────────────────────────────────────────
# 2: Підготовка train/validation split
# Той самий training_set.jsonl що і в експерименті 1
# 80/20 стратифікований split по urgency
# ────────────────────────────────────────────────────────────
import random
random.seed(42)
 
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    all_train = [json.loads(line) for line in f]
 
print(f"Загалом прикладів: {len(all_train)}")
 
# Стратифікований split — берем 20% з кожного urgency рівня
by_urgency = defaultdict(list)
for item in all_train:
    by_urgency[item['expected']['urgency']].append(item)
 
train_split = []
val_split   = []
 
for urgency, items in by_urgency.items():
    random.shuffle(items)
    val_size = max(1, int(len(items) * 0.2))
    val_split.extend(items[:val_size])
    train_split.extend(items[val_size:])
 
print(f"\n📊 SPLIT РЕЗУЛЬТАТ:")
print(f"   Training   : {len(train_split)} прикладів")
print(f"   Validation : {len(val_split)} прикладів")
 
train_urg = Counter(item['expected']['urgency'] for item in train_split)
val_urg   = Counter(item['expected']['urgency'] for item in val_split)
print(f"\n   Training urgency  : {dict(train_urg)}")
print(f"   Validation urgency: {dict(val_urg)}")
 
# Hash-check — val не перетинається з eval set
eval_hashes_path = '/kaggle/working/lesson17_finetune/eval_hashes.json'
if os.path.exists(eval_hashes_path):
    with open(eval_hashes_path) as f:
        eval_hashes = set(json.load(f).keys())
    overlaps = sum(1 for item in val_split
                   if hashlib.md5(item['email'].strip().encode()).hexdigest() in eval_hashes)
    print(f"\n   Overlap val↔eval  : {overlaps} (має бути 0)")
else:
    print(f"\n   ⚠️ eval_hashes.json не знайдено — пропускаємо hash-check")


Загалом прикладів: 300

📊 SPLIT РЕЗУЛЬТАТ:
   Training   : 241 прикладів
   Validation : 59 прикладів

   Training urgency  : {'critical': 50, 'high': 82, 'low': 60, 'medium': 49}
   Validation urgency: {'critical': 12, 'high': 20, 'low': 15, 'medium': 12}

   ⚠️ eval_hashes.json не знайдено — пропускаємо hash-check


In [6]:
# ────────────────────────────────────────────────────────────
# 3: Завантаження моделі через Unsloth
# ~3-5 хвилин
# ────────────────────────────────────────────────────────────
MODEL_NAME     = 'meta-llama/Llama-3.1-8B'
MAX_SEQ_LENGTH = 1024
LORA_R         = 16
LORA_ALPHA     = 32
 
print(f"⏳ Завантажуємо {MODEL_NAME}...")
 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
 
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
 
total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Модель завантажена!")
print(f"   Навчальних: {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   GPU RAM   : {torch.cuda.memory_allocated()/1024**3:.1f} GB")
 


⏳ Завантажуємо meta-llama/Llama-3.1-8B...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Модель завантажена!
   Навчальних: 41,943,040 (0.90%)
   GPU RAM   : 5.8 GB


In [7]:
# ────────────────────────────────────────────────────────────
# 4: Підготовка датасетів
# ────────────────────────────────────────────────────────────
def format_example(item):
    prompt    = build_prompt(item['email'])
    output    = json.dumps(item['expected'], ensure_ascii=False)
    full_text = prompt + " " + output + tokenizer.eos_token
    return {"text": full_text}
 
train_dataset = Dataset.from_list([format_example(item) for item in train_split])
val_dataset   = Dataset.from_list([format_example(item) for item in val_split])
 
print(f"✅ Training dataset  : {len(train_dataset)} прикладів")
print(f"✅ Validation dataset: {len(val_dataset)} прикладів")
 


✅ Training dataset  : 241 прикладів
✅ Validation dataset: 59 прикладів


In [8]:
# ────────────────────────────────────────────────────────────
# 5: Fine-tuning з validation + early stopping
# ~20-35 хвилин
# ────────────────────────────────────────────────────────────
EPOCHS        = 5       # більше epochs — early stopping зупинить вчасно
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
LEARNING_RATE = 1e-4    # менший LR для стабільнішого навчання
 
OUTPUT_DIR = os.path.join(BASE_DIR, 'checkpoints')
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
trainer = UnslothTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = val_dataset,      # ← validation set
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    args = UnslothTrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LEARNING_RATE,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        eval_strategy               = "epoch",    # eval після кожного epoch
        save_strategy               = "no",
        load_best_model_at_end      = False,       # Unsloth обмеження
        warmup_steps                = 10,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        seed                        = 42,
        report_to                   = "none",
    ),
)
 
print(f"🚀 Fine-tuning з validation set...")
print(f"   Epochs    : {EPOCHS} (з eval після кожного)")
print(f"   Train size: {len(train_dataset)}")
print(f"   Val size  : {len(val_dataset)}")
print(f"   LR        : {LEARNING_RATE} (менший ніж v1)\n")
 
trainer_stats = trainer.train()
 
print(f"\n✅ Fine-tuning завершено!")
print(f"   Час : {trainer_stats.metrics['train_runtime']/60:.1f} хв")
print(f"   Loss: {trainer_stats.metrics['train_loss']:.4f}")
 
# Показуємо train vs val loss по epochs
if hasattr(trainer.state, 'log_history'):
    print(f"\n📊 LOSS ПО EPOCHS:")
    print(f"   {'Epoch':<8} {'Train Loss':<14} {'Val Loss':<12}")
    print(f"   {'-'*35}")
    train_logs = [l for l in trainer.state.log_history if 'loss' in l and 'eval_loss' not in l]
    eval_logs  = [l for l in trainer.state.log_history if 'eval_loss' in l]
    for el in eval_logs:
        epoch = el.get('epoch', '?')
        # Знаходимо найближчий train loss
        tl = next((l['loss'] for l in reversed(train_logs)
                   if l.get('epoch',0) <= epoch), '—')
        print(f"   {epoch:<8.1f} {tl if isinstance(tl,str) else f'{tl:.4f}':<14} {el['eval_loss']:.4f}")
 
 


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/241 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/59 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Fine-tuning з validation set...
   Epochs    : 5 (з eval після кожного)
   Train size: 241
   Val size  : 59
   LR        : 0.0001 (менший ніж v1)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 241 | Num Epochs = 5 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.903970,0.633798
2,0.523868,0.504406
3,0.425793,0.475311
4,0.379353,0.477378
5,0.355341,0.480583



✅ Fine-tuning завершено!
   Час : 19.5 хв
   Loss: 0.6521

📊 LOSS ПО EPOCHS:
   Epoch    Train Loss     Val Loss    
   -----------------------------------
   1.0      0.9040         0.6338
   2.0      0.5239         0.5044
   3.0      0.4258         0.4753
   4.0      0.3794         0.4774
   5.0      0.3553         0.4806


In [9]:
# ────────────────────────────────────────────────────────────
# 6: Збереження адаптера v2
# ────────────────────────────────────────────────────────────
ADAPTER_V2 = os.path.join(BASE_DIR, 'lora_adapter_v2')
os.makedirs(ADAPTER_V2, exist_ok=True)
 
model.save_pretrained(ADAPTER_V2)
tokenizer.save_pretrained(ADAPTER_V2)
 
total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(ADAPTER_V2)
    for f in files
)
print(f"✅ Адаптер v2 збережено → {ADAPTER_V2}")
print(f"   Розмір: {total_size/1024/1024:.1f} MB")
 
 


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lesson17_finetune_v2/lora_adapter_v2/tokenizer_config.json.


✅ Адаптер v2 збережено → /kaggle/working/lesson17_finetune_v2/lora_adapter_v2
   Розмір: 176.5 MB


In [10]:
# ────────────────────────────────────────────────────────────
# 7: Re-evaluation v2 на eval set
# ~5-10 хвилин
# ────────────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)
 
with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    eval_data = [json.loads(line) for line in f]
 
results_v2   = []
total_tokens = 0
 
print(f"🚀 Evaluation v2 на {len(eval_data)} прикладах...\n")
 
for i, item in enumerate(eval_data, 1):
    prompt    = build_prompt(item['email'])
    inputs    = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 256,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
 
    new_tokens   = outputs[0][input_len:]
    raw          = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tokens       = len(new_tokens)
    pred         = parse_json_response(raw)
    scores       = score_prediction(pred, item['expected'])
    total_tokens += tokens
 
    results_v2.append({
        'id': item['id'], 'raw': raw, 'parsed': pred,
        'expected': item['expected'], 'scores': scores, 'tokens': tokens,
    })
 
    status     = '✅' if scores['json_valid']      else '❌'
    urgency_ok = '✅' if scores['urgency_correct'] else '❌'
    pred_urg   = pred.get('urgency','—') if pred else 'NONE'
    exp_urg    = item['expected']['urgency']
    print(f"  {i:2}/30 {item['id']} | JSON:{status} | urgency:{urgency_ok} "
          f"| pred:{pred_urg:8} | expected:{exp_urg}")
 
with open(os.path.join(BASE_DIR, 'results_v2.json'), 'w') as f:
    json.dump(results_v2, f, ensure_ascii=False, indent=2)
 


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


🚀 Evaluation v2 на 30 прикладах...



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   1/30 eval_001 | JSON:✅ | urgency:✅ | pred:critical | expected:critical


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  27/30 eval_027 | JSON:✅ | urgency:✅ | pred:high     | expected:high


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  28/30 eval_028 | JSON:✅ | urgency:❌ | pred:medium   | expected:low


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  29/30 eval_029 | JSON:✅ | urgency:✅ | pred:critical | expected:critical
  30/30 eval_030 | JSON:✅ | urgency:✅ | pred:low      | expected:low


In [11]:
# ────────────────────────────────────────────────────────────
# КЛІТИНКА 8: Фінальне порівняння v1 vs v2
# ────────────────────────────────────────────────────────────
n      = len(results_v2)
totals = defaultdict(int)
for r in results_v2:
    for k, v in r['scores'].items():
        totals[k] += v
 
metrics_v2 = {k: round(totals[k]/n*100,1)
              for k in ['json_valid','exact_match','sender_correct',
                        'product_correct','category_correct',
                        'urgency_correct','summary_present']}
metrics_v2['avg_tokens'] = round(total_tokens/n, 1)
 
# Завантажуємо метрики v1
ft_v1_path = '/kaggle/working/lesson17_finetune/ft_metrics.json'
if os.path.exists(ft_v1_path):
    with open(ft_v1_path) as f:
        mv1 = json.load(f)
else:
    # Результати з нашого експерименту 1
    mv1 = {
        "json_valid": 100.0, "exact_match": 56.7,
        "sender_correct": 100.0, "product_correct": 76.7,
        "category_correct": 93.3, "urgency_correct": 76.7,
        "summary_present": 100.0, "avg_tokens": 56.1
    }
 
metrics_order = [
    ('json_valid',        'json_valid'),
    ('exact_match',       'exact_match'),
    ('sender accuracy',   'sender_correct'),
    ('product accuracy',  'product_correct'),
    ('category accuracy', 'category_correct'),
    ('urgency accuracy',  'urgency_correct'),
    ('summary present',   'summary_present'),
]
 
print(f"\n{'='*70}")
print(f"  ПОРІВНЯННЯ: FT v1 (без val) vs FT v2 (з val + medium)")
print(f"{'='*70}")
print(f"  {'Метрика':<22} {'FT v1':>8} {'FT v2':>8} {'Δ':>8}")
print(f"  {'-'*58}")
 
for label, key in metrics_order:
    v1  = mv1.get(key, mv1.get(key.replace('_correct','_accuracy'), 0))
    v2  = metrics_v2[key]
    d   = v2 - v1
    arr = '↑' if d > 0 else ('↓' if d < 0 else '→')
    print(f"  {label:<22} {v1:>7.1f}% {v2:>7.1f}% {arr}{abs(d):>5.1f}%")
 
print(f"  {'-'*58}")
print(f"  {'avg tokens':<22} {mv1.get('avg_tokens',0):>8.1f} {metrics_v2['avg_tokens']:>8.1f}")
print(f"{'='*70}")
 
# Urgency детально — фокус на medium
print(f"\n📊 URGENCY ДЕТАЛЬНО — v2:")
urgency_by_level = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in results_v2:
    exp = r['expected']['urgency']
    urgency_by_level[exp]['total'] += 1
    if r['scores']['urgency_correct']:
        urgency_by_level[exp]['correct'] += 1
 
# Порівняння з v1
v1_urgency = {'critical': '6/7', 'high': '8/9', 'medium': '1/4', 'low': '8/10'}
print(f"  {'Level':<12} {'v1':>8} {'v2':>12}")
print(f"  {'-'*35}")
for level in ['critical', 'high', 'medium', 'low']:
    d   = urgency_by_level[level]
    pct = d['correct']/d['total']*100 if d['total'] > 0 else 0
    print(f"  {level:<12} {v1_urgency.get(level,'?'):>8} {d['correct']}/{d['total']}={pct:.0f}%")
 
# Зберігаємо
with open(os.path.join(BASE_DIR, 'metrics_v2.json'), 'w') as f:
    json.dump({'model': 'Llama-3.1-8B-finetuned-v2', **metrics_v2}, f, indent=2)
 
print(f"\n✅ Експеримент 2 завершено!")
print(f"   Артефакти збережено в {BASE_DIR}")
 



  ПОРІВНЯННЯ: FT v1 (без val) vs FT v2 (з val + medium)
  Метрика                   FT v1    FT v2        Δ
  ----------------------------------------------------------
  json_valid               100.0%   100.0% →  0.0%
  exact_match               56.7%    53.3% ↓  3.4%
  sender accuracy          100.0%   100.0% →  0.0%
  product accuracy          76.7%    76.7% →  0.0%
  category accuracy         93.3%    93.3% →  0.0%
  urgency accuracy          76.7%    73.3% ↓  3.4%
  summary present          100.0%   100.0% →  0.0%
  ----------------------------------------------------------
  avg tokens                 56.1     54.9

📊 URGENCY ДЕТАЛЬНО — v2:
  Level              v1           v2
  -----------------------------------
  critical          6/7 6/7=86%
  high              8/9 6/9=67%
  medium            1/4 2/4=50%
  low              8/10 8/10=80%

✅ Експеримент 2 завершено!
   Артефакти збережено в /kaggle/working/lesson17_finetune_v2
